# train


In [ ]:
from pathlib import Path
import sys
_root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'AGENTS.md').is_file())
if str(_root) not in sys.path: sys.path.insert(0, str(_root))
if str(_root / '.runtime') not in sys.path: sys.path.insert(0, str(_root / '.runtime'))
__file__ = str(_root / 'train.py')
__package__ = ''
__name__ = 'train'


In [ ]:
import argparse
import os
import time
import random
import json
import shutil
from pathlib import Path
import numpy as np

import torch
import torch.nn.functional as F
import torch.optim as optim
from tqdm import tqdm

from utils.dataset import get_loader
from model import myModel
from datetime import datetime
from utils.metrics import Evaluator
from utils.loss_funcs import (
    EdgeAwareLoss,
    SSIMLoss,
    L1_Charbonnier_loss,
    PerceptualLoss,
)
from utils.CIDNet import CIDNet




In [ ]:
def replace_with_retry(source, target):
    # Windows readers may briefly deny replacement while the dashboard reads a file.
    for attempt in range(20):
        try:
            source.replace(target)
            return
        except PermissionError:
            if attempt == 19:
                raise
            time.sleep(0.05)




In [ ]:
def seed_everything(seed):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False




In [ ]:
class Trainer(object):
    def __init__(self, args):
        """
        Initialize the Trainer with arguments from the command line or defaults.
        """
        self.args = args
        self.evaluator = Evaluator()

        self.deep_model = myModel(
            in_channels=3, feature_channels=32, use_white_balance=True
        )
        self.deep_model = self.deep_model.to("cuda")

        self.hvi_net = CIDNet().cuda()
        pth = r"utils/CIDNet_weight_LOLv2_bestSSIM.pth"
        self.hvi_net.load_state_dict(torch.load(pth, map_location="cuda", weights_only=True))
        self.hvi_net.eval()
        self.hvi_net.requires_grad_(False)

        # Create a directory to save model weights, organized by timestamp.
        now_str = datetime.now().strftime("%Y%m%d_%H%M%S")
        self.model_save_path = os.path.join(
            args.save_path, args.model_name, args.dataset, now_str
        )

        if not os.path.exists(self.model_save_path):
            os.makedirs(self.model_save_path)
        with open(os.path.join(self.model_save_path, "config.json"), "w") as f:
            json.dump({**vars(args), "initialization": "resume" if args.resume or args.continue_from else "random", "seed": 7, "torch": torch.__version__, "cuda": torch.version.cuda, "gpu": torch.cuda.get_device_name(0)}, f, indent=2)

        if args.resume is not None:
            if not os.path.isfile(args.resume):
                raise RuntimeError("=> no checkpoint found at '{}'".format(args.resume))
            checkpoint = torch.load(args.resume)
            model_dict = {}
            state_dict = self.deep_model.state_dict()
            for k, v in checkpoint.items():
                if k in state_dict:
                    model_dict[k] = v
            state_dict.update(model_dict)
            self.deep_model.load_state_dict(state_dict)

        self.optim = optim.AdamW(
            self.deep_model.parameters(),
            lr=args.lr,
            weight_decay=args.weight_decay,
            betas=(0.9, 0.999),
        )
        if args.scheduler == "cosine":
            self.scheduler = optim.lr_scheduler.CosineAnnealingLR(
                self.optim, args.epoch, eta_min=args.lr * 1e-4
            )
        else:
            self.scheduler = optim.lr_scheduler.StepLR(
                self.optim, step_size=args.decay_epoch, gamma=args.decay_rate
            )
        #### ALL DATASET RESIZED TO 256x256 ####
        if args.dataset == "EUVP-d":  # 256x256
            args.train_root = f"{args.data_root}/EUVP-Dark/train/"
            args.val_root = f"{args.data_root}/EUVP-Dark/val/"
            args.datasize = 256
            args.resize = False
        elif args.dataset == "EUVP-s":  # 320x240
            args.train_root = f"{args.data_root}/EUVP-Scene/train/"
            args.val_root = f"{args.data_root}/EUVP-Scene/val/"
            args.datasize = 256  # while validating, don't resize
            args.resize = True
        elif args.dataset == "UIEB":  # 1280x~800 => 256x256
            args.train_root = f"{args.data_root}/UIEB/train/"
            args.val_root = f"{args.data_root}/UIEB/val/"
            args.datasize = 256
            args.resize = True
        elif args.dataset == "UFO":  # 320x240
            args.train_root = f"{args.data_root}/UFO-120/train/"
            args.val_root = f"{args.data_root}/UFO-120/val/"
            args.datasize = 256  # while validating, don't resize
            args.resize = True
        elif args.dataset == "LSUI":  # 720x405 => 256x256
            args.train_root = f"{args.data_root}/LSUI/train/"
            args.val_root = f"{args.data_root}/LSUI/val/"
            args.datasize = 256
            args.resize = True

        ## loss functions
        self.vggL = PerceptualLoss()
        self.L1L = L1_Charbonnier_loss()
        self.ssimL = SSIMLoss(device="cuda", window_size=5)
        self.edgeL = EdgeAwareLoss(loss_type="l2", device="cuda")
        self.start_epoch = 1
        self.best_psnr = 0.0
        if args.continue_from:
            saved = torch.load(args.continue_from, map_location="cuda", weights_only=False)
            self.deep_model.load_state_dict(saved["model"])
            self.optim.load_state_dict(saved["optimizer"])
            self.scheduler.load_state_dict(saved["scheduler"])
            self.start_epoch = saved["epoch"] + 1
            self.best_psnr = saved["best_psnr"]
            for name in ("best_model.pth", "initial_model.pth", "records.txt"):
                previous = Path(args.continue_from).parent / name
                if previous.is_file():
                    shutil.copy2(previous, Path(self.model_save_path) / name)
            random.setstate(saved["python_rng"])
            np.random.set_state(saved["numpy_rng"])
            torch.set_rng_state(saved["torch_rng"].cpu())
            torch.cuda.set_rng_state_all([state.cpu() for state in saved["cuda_rng"]])
        else:
            torch.save(self.deep_model.state_dict(), os.path.join(self.model_save_path, "initial_model.pth"))

    def write_status(self, state, epoch=0, **values):
        self.status.update(state=state, epoch=epoch, **values)
        target = Path(self.args.save_path) / "training_status.json"
        temporary = target.with_suffix(".tmp")
        temporary.write_text(json.dumps(self.status), encoding="utf-8")
        try:
            replace_with_retry(temporary, target)
        except PermissionError:
            print("Warning: dashboard status file is busy; training continues.", flush=True)

    def training(self):
        """
        Main training loop that iterates over the training dataset for several steps (max_iters).
        Prints intermediate losses and evaluates on holdout dataset periodically.
        """
        best_psnr = self.best_psnr
        best_round = []
        torch.cuda.empty_cache()
        train_data_loader = get_loader(
            self.args.train_root,
            self.args.train_batch_size,
            self.args.datasize,
            train=True,
            resize=self.args.resize,
            num_workers=self.args.num_workers,
            shuffle=True,
            pin_memory=True,
        )
        self.deep_model.train()
        self.status = {"dataset": self.args.dataset, "epochs": self.args.epoch, "run_path": self.model_save_path}
        if not len(train_data_loader):
            raise RuntimeError("Training dataset is empty")
        for epoch in range(self.start_epoch, self.args.epoch + 1):
            loop = tqdm(
                enumerate(train_data_loader), total=len(train_data_loader), leave=False
            )
            loss_mean = 0.0
            self.optim.zero_grad(set_to_none=True)
            self.write_status("training", epoch, detail="Random initialization; full training split.")
            for step, (x, label, _) in loop:
                x = x.to("cuda")
                label = label.to("cuda")
                pred = self.deep_model(x)

                with torch.no_grad():
                    label_hvi = self.hvi_net.trans.HVIT(label)
                pred_hvi = self.hvi_net.trans.HVIT(pred.clamp(0.0, 1.0))

                hvi_loss = self.L1L(pred_hvi, label_hvi)
                l1_loss = self.L1L(pred, label)
                vgg_loss = self.vggL(pred, label)
                ssim_loss = self.ssimL(pred, label)
                edge_loss = self.edgeL(pred, label)
                final_loss = (
                    l1_loss
                    + 0.5 * hvi_loss
                    + 0.1 * ssim_loss
                    + 0.1 * vgg_loss
                    + 0.1 * edge_loss
                )
                # print(l1_loss.item(), hvi_loss.item(), ssim_loss.item(), vgg_loss.item(), edge_loss.item())

                loss_mean += final_loss.item()

                group_start = (step // self.args.accumulation_steps) * self.args.accumulation_steps
                group_size = min(self.args.accumulation_steps, len(train_data_loader) - group_start)
                (final_loss / group_size).backward()
                if (step + 1) % self.args.accumulation_steps == 0 or step + 1 == len(train_data_loader):
                    self.optim.step()
                    self.optim.zero_grad(set_to_none=True)
                if step % 20 == 0:
                    self.write_status("training", epoch, loss=final_loss.item(), step=step + 1, steps=len(train_data_loader), detail=f"Batch {step + 1}/{len(train_data_loader)}; batch size {self.args.train_batch_size}, accumulation {self.args.accumulation_steps}.")
                loop.set_description(f"[{epoch}/{self.args.epoch}]")
                loop.set_postfix(loss=final_loss.item())

            print(
                f"[{epoch}/{self.args.epoch}], avg. loss is {loss_mean / len(train_data_loader)}, learning rate is {self.optim.param_groups[0]['lr']}"
            )
            if epoch % self.args.epoch_val == 0:
                self.write_status("validating", epoch, loss=loss_mean / len(train_data_loader))
                self.deep_model.eval()
                ssim_, psnr_ = self.validation()
                self.write_status("validating", epoch, psnr=psnr_, ssim=ssim_)

                if psnr_ > best_psnr:
                    best_path = Path(self.model_save_path) / "best_model.pth"
                    torch.save(
                        self.deep_model.state_dict(),
                        best_path.with_suffix(".tmp"),
                    )
                    replace_with_retry(best_path.with_suffix(".tmp"), best_path)
                    best_psnr = psnr_
                    best_round = {
                        "best epoch": epoch,
                        "best PSNR": best_psnr,
                        "best SSIM": ssim_,
                    }
                    with open(
                        os.path.join(self.model_save_path, "records.txt"), "a"
                    ) as f:
                        str_ = "## best round ##\n"
                        for k, v in best_round.items():
                            str_ += f"{k}: {v}. "
                        str_ += "\n####################################"
                        f.write(str_ + "\n")
                with open(os.path.join(self.model_save_path, "records.txt"), "a") as f:
                    str_ = f"[epoch: {epoch}], PSNR: {psnr_}, SSIM: {ssim_}"
                    f.write(str_ + "\n")
                self.deep_model.train()
            self.scheduler.step()
            latest = Path(self.model_save_path) / "last_checkpoint.pth"
            temporary = latest.with_suffix(".tmp")
            torch.save({"model": self.deep_model.state_dict(), "optimizer": self.optim.state_dict(), "scheduler": self.scheduler.state_dict(), "epoch": epoch, "best_psnr": best_psnr, "python_rng": random.getstate(), "numpy_rng": np.random.get_state(), "torch_rng": torch.get_rng_state(), "cuda_rng": torch.cuda.get_rng_state_all()}, temporary)
            replace_with_retry(temporary, latest)
            torch.save(self.deep_model.state_dict(), Path(self.model_save_path) / "last_model.pth")

        print("The accuracy of the best round is ", best_round)
        self.write_status("complete", self.args.epoch, detail="All requested epochs finished; checkpoints saved.")

    def validation(self):
        self.evaluator.reset()
        val_data_loader = get_loader(
            self.args.val_root,
            self.args.eval_batch_size,
            self.args.datasize,
            train=False,
            resize=self.args.resize,
            num_workers=self.args.num_workers,
            shuffle=False,
            pin_memory=True,
        )
        torch.cuda.empty_cache()

        with torch.no_grad():
            loop = tqdm(
                enumerate(val_data_loader), total=len(val_data_loader), leave=False
            )
            for _, (x, label, _) in loop:
                x = x.to("cuda")
                label = (
                    label.numpy().astype(np.float32).transpose(0, 2, 3, 1)
                )  # B, H, W, C
                pred = self.deep_model(x)
                pred = torch.clamp(pred, 0.0, 1.0)
                pred = (
                    pred.data.cpu().numpy().astype(np.float32).transpose(0, 2, 3, 1)
                )  # B, H, W, C

                self.evaluator.evaluation(pred, label)
                loop.set_description("[Validation]")
        ssim_, psnr_ = self.evaluator.getMean()

        print("[Validation] SSIM: %.4f, PSNR: %.4f" % (ssim_, psnr_))
        return ssim_, psnr_




In [ ]:
def main():
    parser = argparse.ArgumentParser()

    parser.add_argument("--epoch", type=int, default=100, help="epoch number")
    parser.add_argument("--epoch_val", type=int, default=1, help="training batch size")
    parser.add_argument("--lr", type=float, default=2e-3, help="learning rate")
    parser.add_argument("--train_batch_size", type=int, default=24)
    parser.add_argument("--eval_batch_size", type=int, default=24)
    parser.add_argument(
        "--decay_rate", type=float, default=0.1, help="decay rate of learning rate"
    )  ##
    parser.add_argument(
        "--decay_epoch", type=int, default=50, help="every n epochs decay learning rate"
    )  ##
    parser.add_argument("--weight_decay", type=float, default=1e-5)
    parser.add_argument("--scheduler", type=str, default="cosine")

    parser.add_argument("--num_workers", type=int, default=4)
    parser.add_argument("--data_root", type=str, default="./UnderWaterDataset")
    parser.add_argument("--accumulation_steps", type=int, default=1)
    parser.add_argument("--continue_from", type=str, help="Full last_checkpoint.pth including optimizer and scheduler")

    parser.add_argument("--dataset", type=str, default="UIEB", choices=["UIEB", "LSUI", 'UFO', 'EUVP-s', 'EUVP-d'])
    parser.add_argument("--model_name", type=str, default="WWE-UIE")
    parser.add_argument("--save_path", type=str, default="./output/")

    parser.add_argument("--resume", type=str)

    args = parser.parse_args()
    if args.accumulation_steps < 1:
        parser.error("--accumulation_steps must be at least 1")
    torch.hub.set_dir(str(Path(__file__).resolve().parent / "downloads/torch"))

    try:
        trainer = Trainer(args)
        trainer.training()
    except Exception as error:
        target = Path(args.save_path) / "training_status.json"
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_text(json.dumps({"state": "failed", "dataset": args.dataset, "epochs": args.epoch, "detail": str(error)}), encoding="utf-8")
        raise


